In [1]:
import sys

In [2]:
sys.path.append("..")
from data.crawl import ShareDistribution
from data.db import *

In [3]:
crawl = ShareDistribution()

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
from itertools import product
import logging

class ShareDistribution:
    def __init__(self):
        self._create_share_distribution_table()
    
    def _create_share_distribution_table(self):
        create_table_sql = '''
        CREATE TABLE IF NOT EXISTS stock_distribution (
            symbol VARCHAR(10) NOT NULL,
            date DATE NOT NULL,
            holding_range VARCHAR(50) NOT NULL,
            holder_count INT NOT NULL,
            share_count BIGINT NOT NULL,
            percentage DECIMAL(10, 2) NOT NULL,
            PRIMARY KEY (symbol, date, holding_range)
        );
        '''
        with get_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(create_table_sql)
                conn.commit()

    def _fetch_html(self, url, session, headers):
        """Fetch HTML content from the given URL."""
        try:
            response = session.get(url, headers=headers)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            logging.error(f"Error fetching HTML: {e}")
            return None

    def _parse_valid_dates(self, soup):
        """Parse valid dates from the HTML soup."""
        return [v.get("value") for v in soup.find_all("option")]

    def _insert_share_data(self, conn, share_data):
        """Insert parsed share data into the database."""
        insert_query = '''
            INSERT INTO stock_distribution 
            (symbol, date, holding_range, holder_count, share_count, percentage)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (symbol, date, holding_range) DO NOTHING;
        '''
        try:
            with conn.cursor() as cursor:
                cursor.execute(insert_query, share_data)
            conn.commit()
        except Exception as e:
            logging.error(f"Error inserting data: {e}")
            conn.rollback()

    def Get(self, symbols=[]):
        headers = {
            'User-Agent':  'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
        }
        
        url = 'https://www.tdcc.com.tw/portal/zh/smWeb/qryStock'
        with requests.session() as session:
            html = self._fetch_html(url, session, headers)
            if not html:
                return
            soup = BeautifulSoup(html, 'lxml')
            valid_dates = self._parse_valid_dates(soup)
        
        with get_connection() as conn:
            for symbol, date in product(symbols, valid_dates):
                print(symbol, date)
                payload = {
                    'scaDate': date,
                    'sqlMethod': 'StockNo',
                    'stockNo': str(symbol),
                    **{e.get('name'): e.get('value') for e in soup.select('input[type=hidden]')}
                }
                
                time.sleep(1)  # Prevent overloading the server
                try:
                    res = session.post(url, headers=headers, data=payload)
                    res.raise_for_status()
                except requests.exceptions.RequestException as e:
                    logging.error(f"Error fetching share data for {symbol} on {date}: {e}")
                    continue
                
                dfs = pd.read_html(res.text)
                share_table = dfs[1]
                
                share_table.columns = ["", "holding_range", "holder_count", "share_count", "percentage"]
                share_table = share_table.loc[:14, share_table.columns[1:]]
                share_table["holding_range"] = share_table["holding_range"].str.split('-').str[1]
                share_table["holding_range"] = share_table["holding_range"].str.replace(',', '')
                share_table.loc[14, "holding_range"] = "999999999"
                share_table["holding_range"] = share_table["holding_range"].astype(int)
                
                for _, row in share_table.iterrows():
                    share_data = (
                        symbol,
                        date,
                        int(row["holding_range"]),
                        int(row["holder_count"]),
                        int(row["share_count"]),
                        float(row["percentage"])
                    )
                    self._insert_share_data(conn, share_data)
                    
            release_connection(conn)


In [5]:
ShareDistribution().Get(["1440"])

1440 20250124


/var/folders/89/jcq7r95d3csfd9ls92wxdzj40000gn/T/ipykernel_86010/917459048.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(res.text)


1440 20250117


/var/folders/89/jcq7r95d3csfd9ls92wxdzj40000gn/T/ipykernel_86010/917459048.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(res.text)


AttributeError: Can only use .str accessor with string values!

In [ ]:
# # data.columns = [""]

# # for _, row in data.iterrows():
# #     # value_replace_str = ("%s,"*6)[:-1]
    
    
# #     row
# data = old
# data.columns = ["", "holding_range", "holder_count", "share_count", "percentage"]
# data = data.loc[:data.index.size-2,data.columns[1:]]
# data["holding_range"] = data["holding_range"].str.split('-').str[1]
# data["holding_range"] = data["holding_range"].str.replace(',', '')
# data.loc[14, "holding_range"] = 999_999_999
# data["holding_range"] = data["holding_range"].astype(int)

# for _, row in data.iterrows():
#     row_dict = row.to_dict()
#     row_dict["symbol"] = '2330'
#     row_dict["date"] = "'2024-01-01'"
#     key_replace = ", ".join(row_dict.keys())
#     val_replace = ", ".join(str(v) for v in row_dict.values())
    
#     break
# sql = f'''
#     INSERT INTO stock_distribution 
#     ({key_replace}) 
#     VALUES ({val_replace});
# '''

# conn = get_connection()
# cursor = conn.cursor()
# cursor.execute(sql)
# conn.commit()
# release_connection(conn)